In [59]:
import numpy as np
import random
from graphviz import Digraph

In [60]:
def _unbroadcast(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, (g, s) in enumerate(zip(grad.shape, shape)):
        if s == 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad

class Tensor:
    def __init__(self, data, _children=(), _op=''):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Tensor(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += _unbroadcast(out.grad, self.data.shape)
            other.grad += _unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data - other.data, (self, other), '-')
        def _backward():
            self.grad += _unbroadcast(out.grad, self.data.shape)
            other.grad += _unbroadcast(out.grad, -other.data.shape)
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += _unbroadcast(other.data * out.grad, self.data.shape)
            other.grad += _unbroadcast(self.data  * out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data @ other.data, (self, other), '@')
        def _backward():
            # d(loss)/dA = d(loss)/dC @ B.T
            # d(loss)/dB = A.T @ d(loss)/dC
            self.grad  += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad
        out._backward = _backward
        return out

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float))
        out = Tensor(self.data ** exponent, (self,), f'**{exponent}')
        def _backward():
            self.grad += exponent * (self.data ** (exponent - 1)) * out.grad
        out._backward = _backward
        return out

    def sum(self):
        """Reduce all elements to a scalar."""
        out = Tensor(self.data.sum(), (self,), 'sum')
        def _backward():
            self.grad += np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), (self,), 'ReLU')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        out = Tensor(np.tanh(self.data), (self,), 'Tanh')
        def _backward():
            self.grad += (1 - out.data**2) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):            return self * -1
    def __sub__(self, other):     return self + (-other)
    def __truediv__(self, other): return self * other**-1
    def __radd__(self, other):    return self + other
    def __rmul__(self, other):    return self * other
    def __rsub__(self, other):    return Tensor(other) + (-self)

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = np.ones_like(self.data)   # seed
        for node in reversed(topo):
            node._backward()



In [61]:
class Linear:
    def __init__(self,nin,nout):
        self.W = Tensor(np.random.randn(nin,nout)*np.sqrt(2.0/nin))
        self.b = Tensor(np.zeros((1,nout)))

    def __call__(self,x):
        return x @ self.W +self.b

    def parameters(self):
        return [self.W,self.b]

class ReLU:
    def __call__(self, x):
        return x.relu()

    def parameters(self):
        return []

class Sequential():
    def __init__(self, *layers):
        self.layers = layers

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


class SGD:
    def __init__(self, params, lr=0.01):
        self.params = params
        self.lr = lr

    def zero_grad(self):
        for p in self.params:
            p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.params:
            p.data -= self.lr * p.grad

class MSELoss:
    def __call__(self, pred, target):
        n = pred.data.shape[0]
        return ((pred - target) ** 2.0).sum()*(1.0/n)


In [62]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def tensor_summary(t):
    if t.data.size <= 4:
        return np.array2string(t.data, precision=2, separator=',')
    else:
        return f"shape={list(t.data.shape)}"

def draw_dot(root, format='svg', rankdir='LR'):
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir})
    nodes, edges = trace(root)

    for n in nodes:
        dot.node(
            name=str(id(n)),
            label="{ %s | data: %s | grad: %s }" % (
                n._op or 'input',
                tensor_summary(n),
                tensor_summary(type('g',(), {'data': n.grad, 'size': n.grad.size})())
                if hasattr(n, 'grad') else '?'),
            shape='record'
        )
        if n._op:
            op_id = str(id(n)) + n._op
            dot.node(name=op_id, label=n._op)
            dot.edge(op_id, str(id(n)))

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [63]:
X = Tensor([[4.0, 3.0], [-2.0, -1.0], [1.0, -1.5], [-2.0, -2.0], [-3.0, -5.0], [4.0, -1.0], [7.0, 9.0], [-1.0,5.0]])
y = Tensor([[1.0], [1.0], [-1.0], [1.0], [1.0], [-1.0], [1.0], [-1.0]])

np.random.seed(56)
model = Sequential(Linear(2,8), ReLU(), Linear(8,1))

optimiser = SGD(model.parameters(), lr=0.001)
criterion = MSELoss()

for step in range(500):
    out = model(X)
    loss = criterion(out, y)

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

    print(f"step {step:2d}  loss={loss.data:.4f}")

step  0  loss=17.0271
step  1  loss=12.6410
step  2  loss=9.9421
step  3  loss=8.2276
step  4  loss=7.1012
step  5  loss=6.3326
step  6  loss=5.7851
step  7  loss=5.3766
step  8  loss=5.0571
step  9  loss=4.7957
step 10  loss=4.5756
step 11  loss=4.3824
step 12  loss=4.2088
step 13  loss=4.0500
step 14  loss=3.9028
step 15  loss=3.7651
step 16  loss=3.6356
step 17  loss=3.5131
step 18  loss=3.3970
step 19  loss=3.2866
step 20  loss=3.1816
step 21  loss=3.0815
step 22  loss=2.9860
step 23  loss=2.8948
step 24  loss=2.8076
step 25  loss=2.7243
step 26  loss=2.6447
step 27  loss=2.5684
step 28  loss=2.4954
step 29  loss=2.4255
step 30  loss=2.3584
step 31  loss=2.2941
step 32  loss=2.2325
step 33  loss=2.1733
step 34  loss=2.1165
step 35  loss=2.0620
step 36  loss=2.0096
step 37  loss=1.9592
step 38  loss=1.9108
step 39  loss=1.8642
step 40  loss=1.8193
step 41  loss=1.7762
step 42  loss=1.7346
step 43  loss=1.6946
step 44  loss=1.6560
step 45  loss=1.6187
step 46  loss=1.5829
step 47  lo

In [64]:
dot = draw_dot(loss)
dot.render('my_graph_pytorch', format='svg', cleanup=True)

'my_graph_pytorch.svg'